# Sistema de Recomendação Trocabook

Este notebook tem como objetivo experimentar e validar uma abordagem de
mineração de dados para recomendação personalizada de livros no Trocabook.

O sistema utiliza as interações realizadas pelos usuários na plataforma:

- PESQUISA
- VISUALIZACAO
- INICIO_CONVERSA

As interações serão utilizadas para construir um perfil de interesse do
usuário e recomendar livros com características semelhantes.

In [19]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [20]:
livros = pd.DataFrame([
    {
        "id": "L1",
        "titulo": "Harry Potter e a Pedra Filosofal",
        "autores": "J. K. Rowling",
        "categorias": "fantasia aventura"
    },
    {
        "id": "L2",
        "titulo": "Harry Potter e a Câmara Secreta",
        "autores": "J. K. Rowling",
        "categorias": "fantasia aventura"
    },
    {
        "id": "L3",
        "titulo": "O Hobbit",
        "autores": "J. R. R. Tolkien",
        "categorias": "fantasia aventura"
    },
    {
        "id": "L4",
        "titulo": "Dom Casmurro",
        "autores": "Machado de Assis",
        "categorias": "romance literatura brasileira"
    },
    {
        "id": "L5",
        "titulo": "Clean Code",
        "autores": "Robert Martin",
        "categorias": "programacao tecnologia"
    }
])

livros

,id,titulo,autores,categorias
0,L1,Harry Potter e a Pedra Filosofal,J. K. Rowling,fantasia aventura
1,L2,Harry Potter e a Câmara Secreta,J. K. Rowling,fantasia aventura
2,L3,O Hobbit,J. R. R. Tolkien,fantasia aventura
3,L4,Dom Casmurro,Machado de Assis,romance literatura brasileira
4,L5,Clean Code,Robert Martin,programacao tecnologia


In [21]:
interacoes = pd.DataFrame([
    {
        "uidUsuario": "U1",
        "tipoInteracao": "VISUALIZACAO",
        "uidLivro": "L1",
        "termoPesquisa": None
    },
    {
        "uidUsuario": "U1",
        "tipoInteracao": "INICIO_CONVERSA",
        "uidLivro": "L1",
        "termoPesquisa": None
    },
    {
        "uidUsuario": "U1",
        "tipoInteracao": "VISUALIZACAO",
        "uidLivro": "L3",
        "termoPesquisa": None
    },
    {
        "uidUsuario": "U1",
        "tipoInteracao": "PESQUISA",
        "uidLivro": None,
        "termoPesquisa": "fantasia"
    }
])

interacoes

,uidUsuario,tipoInteracao,uidLivro,termoPesquisa
0,U1,VISUALIZACAO,L1,NaN
1,U1,INICIO_CONVERSA,L1,NaN
2,U1,VISUALIZACAO,L3,NaN
3,U1,PESQUISA,NaN,fantasia


In [22]:
pesos = {
    "PESQUISA": 0.5,
    "VISUALIZACAO": 1,
    "INICIO_CONVERSA": 3
}


In [23]:
interacoes["peso"] = interacoes["tipoInteracao"].map(pesos)

interacoes

,uidUsuario,tipoInteracao,uidLivro,termoPesquisa,peso
0,U1,VISUALIZACAO,L1,NaN,1.0
1,U1,INICIO_CONVERSA,L1,NaN,3.0
2,U1,VISUALIZACAO,L3,NaN,1.0
3,U1,PESQUISA,NaN,fantasia,0.5


In [24]:
vectorizer_categorias = TfidfVectorizer()
vectorizer_autores = TfidfVectorizer()

matriz_categorias = vectorizer_categorias.fit_transform(
    livros["categorias"]
)

matriz_autores = vectorizer_autores.fit_transform(
    livros["autores"]
)

In [25]:
similaridade_categorias = cosine_similarity(matriz_categorias)
similaridade_autores = cosine_similarity(matriz_autores)

In [26]:
import math

PESO_CATEGORIA = 0.75
PESO_AUTOR = 0.25

assert math.isclose(
    PESO_CATEGORIA + PESO_AUTOR,
    1.0
), "A soma dos pesos deve ser igual a 1"

similaridades = (
    similaridade_categorias * PESO_CATEGORIA +
    similaridade_autores * PESO_AUTOR
)

In [27]:
matriz_similaridade = pd.DataFrame(
    similaridades,
    index=livros["titulo"],
    columns=livros["titulo"]
)

matriz_similaridade

titulo,Harry Potter e a Pedra Filosofal,Harry Potter e a Câmara Secreta,O Hobbit,Dom Casmurro,Clean Code
titulo,,,,,
Harry Potter e a Pedra Filosofal,1.00,1.00,0.75,0.0,0.0
Harry Potter e a Câmara Secreta,1.00,1.00,0.75,0.0,0.0
O Hobbit,0.75,0.75,1.00,0.0,0.0
Dom Casmurro,0.00,0.00,0.00,1.0,0.0
Clean Code,0.00,0.00,0.00,0.0,1.0


In [28]:
def obter_interacoes_usuario(interacoes, uid_usuario, tipo_interacao=None):
    resultado = interacoes[
        interacoes["uidUsuario"] == uid_usuario
    ]

    if tipo_interacao is not None:
        resultado = resultado[
            resultado["tipoInteracao"] == tipo_interacao
        ]

    return resultado

In [29]:
def obter_interacoes_livros(interacoes_usuario):
    return interacoes_usuario[
        interacoes_usuario["uidLivro"].notna()
    ]

In [30]:
def calcular_interesse_por_livro(interacoes_livros):
    return (
        interacoes_livros
        .groupby("uidLivro")["peso"]
        .sum()
        .reset_index()
    )

In [31]:
def calcular_scores_livros(
    livros,
    interesses,
    similaridades
):
    scores = np.zeros(len(livros))

    for _, interesse in interesses.iterrows():
        uid_livro = interesse["uidLivro"]
        peso = interesse["peso"]

        indice = livros.index[
            livros["id"] == uid_livro
        ][0]

        scores += similaridades[indice] * peso

    return scores

In [32]:
def calcular_scores_pesquisas(
    pesquisas,
    livros,
    vectorizer_categorias,
    matriz_categorias
):
    scores = np.zeros(len(livros))

    for _, pesquisa in pesquisas.iterrows():
        vetor_pesquisa = vectorizer_categorias.transform(
            [pesquisa["termoPesquisa"]]
        )

        similaridade = cosine_similarity(
            vetor_pesquisa,
            matriz_categorias
        )[0]

        scores += similaridade * pesquisa["peso"]

    return scores

In [33]:
def montar_ranking(livros, scores, livros_interagidos):
    ranking = livros.copy()
    ranking["score"] = scores

    ranking = ranking[
        ~ranking["id"].isin(livros_interagidos)
    ]

    ranking = ranking[
        ranking["score"] > 0
    ]

    return ranking.sort_values(
        by="score",
        ascending=False
    )

In [34]:
def recomendar_livros(
    uid_usuario,
    livros,
    interacoes,
    similaridades,
    vectorizer_categorias,
    matriz_categorias
):
    interacoes_usuario = obter_interacoes_usuario(
        interacoes,
        uid_usuario
    )

    interacoes_livros = obter_interacoes_livros(
        interacoes_usuario
    )

    pesquisas = obter_interacoes_usuario(
        interacoes,
        uid_usuario,
        "PESQUISA"
    )

    interesses = calcular_interesse_por_livro(
        interacoes_livros
    )

    scores_livros = calcular_scores_livros(
        livros,
        interesses,
        similaridades
    )

    scores_pesquisas = calcular_scores_pesquisas(
        pesquisas,
        livros,
        vectorizer_categorias,
        matriz_categorias
    )

    scores_finais = scores_livros + scores_pesquisas

    livros_interagidos = interesses["uidLivro"].tolist()

    return montar_ranking(
        livros,
        scores_finais,
        livros_interagidos
    )

In [35]:
recomendacoes = recomendar_livros(
    "U1",
    livros,
    interacoes,
    similaridades,
    vectorizer_categorias,
    matriz_categorias
)

recomendacoes[["id", "titulo", "score"]]

,id,titulo,score
1,L2,Harry Potter e a Câmara Secreta,5.103553
